In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("/notebooks/"))

import torch
from pathlib import Path
from src.tokenizer import INSTR_OFF, INSTR_NAMES, BOS, EOS, BAR, KEY_OFF, TIME_OFF, TEMPO_OFF, tokens_to_score
from src.model import load_model, CHECKPOINT_DIR

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)
INFILL_WEIGHTS_FILE = CHECKPOINT_DIR / "infill_model.pt"

print("Available instruments:")
for i, name in enumerate(INSTR_NAMES):
    print(f"  {i:2d}: {name}")

Available instruments:
   0: other
   1: piano
   2: harpsichord
   3: organ
   4: accordion
   5: guitar
   6: bass_guitar
   7: violin
   8: viola
   9: cello
  10: double_bass
  11: harp
  12: flute
  13: oboe
  14: clarinet
  15: bassoon
  16: saxophone
  17: horn
  18: trumpet
  19: trombone
  20: tuba
  21: percussion
  22: timpani
  23: celesta
  24: voice_soprano
  25: voice_mezzo
  26: voice_alto
  27: voice_tenor
  28: voice_baritone
  29: voice_bass
  30: voice
  31: choir


### Generation Parameters

In [12]:
# Instruments to include in the generated score (names from INSTR_NAMES above)
INSTRUMENTS = ["voice", "piano"]

# Number of measures to generate
N_MEASURES = 24

# Key signature: number of sharps (-6=6 flats, 0=C major, 6=6 sharps)
KEY_SHARPS = 0      # 0 = C major / A minor

# Time signature index into [2/4, 3/4, 4/4, 6/8, 3/8, 12/8]
TIME_SIG_IDX = 2    # 2 = 4/4

# Tempo bucket index into [20,40,52,63,72,84,96,108,116,126,138,152,168,192,216,256] BPM ranges
TEMPO_IDX = 5       # 8 = 116-126 BPM (~Moderato)

# Sampling parameters
TEMPERATURE = 0.9
TOP_K       = 50
MAX_NEW     = 1500

### Generate

In [13]:
# Resolve instrument indices and build header tokens
instr_indices = []
for name in INSTRUMENTS:
    name_lower = name.lower()
    if name_lower in INSTR_NAMES:
        instr_indices.append(INSTR_NAMES.index(name_lower))
    else:
        print(f"Unknown instrument '{name}', falling back to 'other'")
        instr_indices.append(0)

header_tokens = [INSTR_OFF + idx for idx in instr_indices]
print(f"Instruments: {INSTRUMENTS}")
print(f"Header tokens: {header_tokens}")

# Build the forced opening bar header: BAR + key sig + time sig + tempo.
# This primes the model with the desired key/meter/tempo before free sampling,
# putting it in-distribution with the standard training examples.
first_bar_prefix = [
    BAR,
    KEY_OFF + (KEY_SHARPS + 6),   # KEY_SHARPS in [-6, 6] → index in [0, 12]
    TIME_OFF + TIME_SIG_IDX,
    TEMPO_OFF + TEMPO_IDX,
]
print(f"First bar prefix tokens: {first_bar_prefix}  (BAR, key={KEY_SHARPS:+d}, time_idx={TIME_SIG_IDX}, tempo_idx={TEMPO_IDX})")

# Load infill model
print(f"\nLoading model from {INFILL_WEIGHTS_FILE}...")
model, cfg = load_model(DEVICE)
if INFILL_WEIGHTS_FILE.exists():
    state = torch.load(INFILL_WEIGHTS_FILE, map_location=DEVICE, weights_only=False)
    model.load_state_dict(state)
    print("Loaded infill weights.")
else:
    print("Infill weights not found — using base model weights.")
print(f"Model loaded ({model.num_params():,} params)")

# Generate using the standard AR format — this matches the 50% standard training
# examples ([BOS]+header+measures+[EOS]) rather than the FIM format with empty
# prefix/suffix which the model has never seen during training.
print(f"\nGenerating {N_MEASURES} measures from scratch...")
fill_tokens = model.generate_standard(
    header_tokens=header_tokens,
    first_bar_prefix=first_bar_prefix,
    target_bars=N_MEASURES,
    max_new=MAX_NEW,
    temperature=TEMPERATURE,
    top_k=TOP_K,
    device=DEVICE,
)

generated_bars = fill_tokens.count(BAR)
print(f"Generated {len(fill_tokens)} tokens ({generated_bars} bars)")

Instruments: ['voice', 'piano']
Header tokens: [461, 432]
First bar prefix tokens: [3, 181, 190, 420]  (BAR, key=+0, time_idx=2, tempo_idx=5)

Loading model from storage\checkpoints\infill_model.pt...
Loaded infill weights.
Model loaded (20,425,728 params)

Generating 24 measures from scratch...
Written 2 bars of 24
Written 3 bars of 24
Written 4 bars of 24
Written 5 bars of 24
Written 6 bars of 24
Written 7 bars of 24
Written 8 bars of 24
Written 9 bars of 24
Written 10 bars of 24
Written 11 bars of 24
Written 12 bars of 24
Written 13 bars of 24
Written 14 bars of 24
Written 15 bars of 24
Written 16 bars of 24
Written 17 bars of 24
Written 18 bars of 24
Generated 1500 tokens (17 bars)


### Export

In [14]:
# Reconstruct score from generated tokens.
# first_bar_prefix contains [BAR, KEY_SIG, TIME_SIG, TEMPO] which is part of the
# prompt in generate_standard(), so it must be prepended to fill_tokens here.
full_tokens = [BOS] + header_tokens + first_bar_prefix + fill_tokens + [EOS]
score = tokens_to_score(full_tokens)

# Color generated notes red for easy identification
for el in score.flatten().notesAndRests:
    if el.isNote or el.isChord:
        el.style.color = 'red'

# Build output filename from instruments and measure count
instr_tag = "_".join(INSTRUMENTS)
name = f"scratch_{instr_tag}_{N_MEASURES}bars"
midi_path = str(OUTPUT_DIR / f"{name}.mid")
xml_path  = str(OUTPUT_DIR / f"{name}.xml")

score.write("midi", fp=midi_path)
print(f"Saved MIDI -> {midi_path}")

try:
    score.makeNotation().write("musicxml", fp=xml_path)
    print(f"Saved MusicXML -> {xml_path}")
except Exception as e:
    print(f"MusicXML export skipped: {e}")
    xml_path = None

note_count   = sum(1 for el in score.flatten().notesAndRests if el.isNote or el.isChord)
measure_count = len(score.parts[0].getElementsByClass("Measure")) if score.parts else 0
print(f"\nScore summary: {measure_count} measures, {note_count} notes/chords")

Saved MIDI -> output\scratch_voice_piano_24bars.mid
Saved MusicXML -> output\scratch_voice_piano_24bars.xml

Score summary: 18 measures, 301 notes/chords
